<a href="https://colab.research.google.com/github/aritrasinha531-ai/silver-invention/blob/main/text_summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
from google.colab import files
uploaded1=files.upload()

In [ ]:
train_data=pd.read_csv("samsum-train.csv")

In [ ]:
val_data=pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head()

In [ ]:
val_data.head()

In [ ]:
train_data["dialogue"][0]

In [ ]:
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [ ]:
#data preprocessing
import re
def clean_data(text):
  text=re.sub(r"\r\n"," ",text) #lines
  text=re.sub(r"\s+"," ",text) #spaces
  text=re.sub(r"<.*?>"," ",text) #HTML
  text=text.strip().lower()
  return text
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)
val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [ ]:
train_data["dialogue"][0]

In [ ]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")
def tokenize(data):
  inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
  targets=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)
  inputs["labels"]=targets["input_ids"]
  return inputs

In [ ]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")
import torch
if torch.backends.mps.is_available():
  device= torch.device("mps")
elif torch.cuda.is_available():
  device= torch.device("cuda")
else:
  device= torch.device("cpu")
print("device=",device)
model.to(device)

In [ ]:
# training arguments
training_args=TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500
)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
# train model
trainer.train()

In [ ]:
# model load=> fine-tune=>save the model
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_model")

In [ ]:
def summarize_dialogue(dialogue):
  dialogue=clean_data(dialogue)

  # tokenize
  inputs=tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"
  )

  # generate summary
  targets=model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      num_beams=4,
      early_stopping=True
  )

  #decoded output
  summary=tokenizer.decode(targets[0],skip_special_tokens=True)
  return summary

In [ ]:
test_dialogue="""Taehyung: Jungkook… can I do something?
Jungkook: Why are you asking like that? You’re making me nervous.

Taehyung: Just trust me.

(Taehyung gently lifts his hand to Jungkook’s cheek, brushing it softly.)

Jungkook: Tae…

(They pause for a moment, both a little shy but smiling.)

Taehyung: If you’re not comfortable, tell me.
Jungkook: …I am.

(Taehyung slowly leans in, giving Jungkook a soft, gentle kiss. It’s brief, but full of warmth.)

(Jungkook looks surprised for a second, then smiles softly.)

Jungkook: You really did that…
Taehyung: Yeah… I’ve wanted to for a while.

(Jungkook leans in this time, placing a small kiss on Taehyung’s lips.)

Jungkook: Me too.

(They both laugh quietly, staying close, foreheads touching.)

Taehyung: We’re really doing this, huh?
Jungkook: Yeah… we are."""

summary=summarize_dialogue(test_dialogue)
print("summary=",summary)

In [ ]:
print("this is text summarizer")